# `load` — cognitive load and vocal arousal

A direction that tests a **theory** instead of a backbone: instead of asking a frozen foundation
model what it sees, compute a small set of acoustic features that the literature says should differ
between truth and deception, and give those to a classifier.

The two theories, their evidence and what this direction can and cannot show are in
[`docs/theory.md`](docs/theory.md). The features, one by one, are in [`docs/features.md`](docs/features.md).

**Rules** (from `lie-detector/AGENTS.md`)

1. Don't touch `data/processed/splits/` or `lib/` — if you need a change there, tell the group.
2. The model lives in this notebook: it is a few hundred parameters, not a module.
3. **Results come from the five folds.** This notebook never touches the test set.
4. Save predictions per clip: they are what makes the directions comparable.

## 0 — Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
if not os.path.isdir("/content/DeepLearningCourse/.git"):
    !git clone -q https://github.com/mattsays/DeepLearningCourse /content/DeepLearningCourse
%cd /content/DeepLearningCourse/lie-detector
!git pull -q
import sys; sys.path.insert(0, '.')

/content/DeepLearningCourse/lie-detector


In [3]:
# Praat bindings. The energy features need nothing, the pitch and perturbation ones need this.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "praat-parselmouth"], check=True)
import parselmouth
print("parselmouth", parselmouth.VERSION)

parselmouth 0.4.7


## 1 — Settings

In [16]:
from pathlib import Path

from lib.dataset import cache_dir

DATA = Path("/content/drive/MyDrive/dolos")
ENCODER = "acoustic-load"          # the folder under cached_features/
CACHE = cache_dir(ENCODER, DATA)   # DATA/cached_features/acoustic-load/<clip_id>.npy

DIRECTION = "load"
SEED = 0
BATCH_SIZE = 32
EPOCHS = 60
PATIENCE = 10
FOLDS = 5
SEEDS = [0, 1, 2]
SPLIT = "official"                  # "subject" (our protocol) or "official" (the paper's 3-fold)

# The features are 17 numbers per clip: CPU is fine, the GPU only matters for consistency with
# the other directions. MPS is for a local Mac.
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("device:", DEVICE)

print(CACHE)

device: cuda
/content/drive/MyDrive/dolos/cached_features/acoustic-load


## 2 — The frozen splits

Same protocol as every other direction: `Protocol.load` verifies the manifest hashes, so a split
edited by hand raises instead of silently giving a different one.

In [17]:
from lib.dataset import Protocol, Clips

folds = Protocol.load(DATA / "splits")
clips = Clips.load(DATA)
print(clips.describe_source())     # post it: same fingerprint = same dataset

N_FOLDS = 3 if SPLIT == "official" else folds.folds


def fold_parts(fold):
    """(train, val, holdout) for the current protocol. The official split has no val."""
    if SPLIT == "official":
        train_ids, test_ids = folds.official(fold)
        return train_ids, None, test_ids
    return folds.subject(fold)


train, val, holdout = fold_parts(1)
for name, ids in (("train", train), ("val", val), ("holdout", holdout)):
    print(f"{name:8}", "(none)" if ids is None else clips.describe(ids))

dataset.csv sha256:8fa198b1fe71  1678 clips  1452 with media
train    {'clips': 944, 'subjects': 150, 'deception': 53.2, 'pos_weight': 0.88, 'female': 20.0, 'majority_baseline': 53.2}
val      (none)
holdout  {'clips': 483, 'subjects': 131, 'deception': 53.6, 'pos_weight': 0.865, 'female': 21.1, 'majority_baseline': 53.6}


## 3 — The acoustic cache

One `.npy` per clip with the 17 features of [`features.py`](features.py), written into the shared
folder like every other encoder — so `CachedFeatures` reads it without knowing what it is.

The loop skips clips already on disk, keeps going when one file is broken, and records the failures.
NaN values (Praat gives up on some clips) are imputed with the column median over the working set:
that uses feature marginals only, never labels.

In [18]:
import sys
from pathlib import Path
import importlib

import numpy as np
import pandas as pd
from tqdm import tqdm


import features

importlib.reload(features)  

from features import FEATURE_NAMES, acoustic_features

print(f"{len(FEATURE_NAMES)} features:", ", ".join(FEATURE_NAMES))

17 features: duration_s, rms_mean, rms_std, energy_range_db, voiced_ratio, pause_count, pause_ratio, pause_mean_ms, onsets_per_s, f0_mean, f0_std, f0_range_st, jitter_local, shimmer_local, hnr_mean, intensity_mean_db, intensity_std_db


In [19]:
working = []
for fold in range(1, FOLDS + 1):
    for part in folds.subject(fold):
        working += part
working = list(dict.fromkeys(working))
working = [c for c in clips.keep(working) if clips[c].audio is not None]
print(f"{len(working)} clips with audio")

CACHE.mkdir(parents=True, exist_ok=True)
todo = [c for c in working if not (CACHE / f"{c}.npy").exists()]
print(f"{len(todo)} to compute, {len(working) - len(todo)} already cached")

failures = {}
for clip_id in tqdm(todo, desc=ENCODER, unit="clip", dynamic_ncols=True):
    try:
        np.save(CACHE / f"{clip_id}.npy", acoustic_features(clips[clip_id].audio))
    except Exception as error:                     # one broken clip must not kill the run
        failures[clip_id] = f"{type(error).__name__}: {error}"[:200]

print(f"{len(failures)} failed")
if failures:
    print("  first:", dict(list(failures.items())[:3]))

1158 clips with audio
0 to compute, 1158 already cached


acoustic-load: 0clip [00:00, ?clip/s]

0 failed


In [20]:
# impute the NaNs Praat leaves behind, with the column median over the working set
present = [c for c in working if (CACHE / f"{c}.npy").exists()]
matrix = np.stack([np.load(CACHE / f"{c}.npy") for c in present])
median = np.nanmedian(matrix, axis=0)

repaired = 0
for index, clip_id in enumerate(present):
    row = matrix[index]
    if np.isnan(row).any():
        np.save(CACHE / f"{clip_id}.npy", np.where(np.isnan(row), median, row))
        repaired += 1

print(f"{repaired} vectors had NaNs, imputed with the column median")
print(f"cache: {len(present)} clips, shape {matrix.shape[1:]}")

0 vectors had NaNs, imputed with the column median
cache: 1158 clips, shape (17,)


## 4 — Load and check

`CachedFeatures` hands back the array as a float32 tensor; `make_loader` collates it. Nothing here
knows the array is a feature vector rather than an embedding.

In [21]:
import torch
from lib.dataset import CachedFeatures, make_loader

have = {p.stem for p in CACHE.glob("*.npy")}


def usable(ids):
    return [c for c in clips.keep(ids) if c in have]


def loader_for(ids, shuffle, seed=SEED):
    dataset = CachedFeatures(usable(ids), clips, CACHE)
    return make_loader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, seed=seed)


train_ids, val_ids, holdout_ids = fold_parts(1)
for name, ids in (("train", train_ids), ("val", val_ids), ("holdout", holdout_ids)):
    if ids is None:
        print(f"{name:8} (the official split has no validation set)")
    else:
        print(f"{name:8} {len(ids):4d} split clips -> {len(usable(ids)):4d} usable")

x, y = next(iter(loader_for(train_ids, shuffle=False)))
print("batch", tuple(x.shape), tuple(y.shape))

train     944 split clips ->  753 usable
val      (the official split has no validation set)
holdout   483 split clips ->  381 usable
batch (32, 17) (32,)


## 5 — The model

Seventeen features and a few hundred parameters, against the 94M frozen parameters of WavLM in
`tia`. That comparison is the point of the direction: if these features reach the same AUC, the
foundation model is not what is doing the work.

The scaler is a buffer, filled from the **training fold only** — so the normalisation never sees the
holdout.

In [22]:
import torch.nn as nn


class LoadHead(nn.Module):
    """Standardise, then a small MLP. The scaler is set per fold, from the training split."""

    def __init__(self, n_features, hidden=32, drop=0.3):
        super().__init__()
        self.register_buffer("mu", torch.zeros(n_features))
        self.register_buffer("sigma", torch.ones(n_features))
        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(n_features, hidden),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden, 1),
        )

    def fit_scaler(self, features):
        self.mu.copy_(features.mean(dim=0))
        self.sigma.copy_(features.std(dim=0).clamp(min=1e-6))

    def forward(self, x):
        return self.head((x - self.mu) / self.sigma).squeeze(-1)


net = LoadHead(len(FEATURE_NAMES))
params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(net)
print(f"load: {params:,} trainable parameters (WavLM alone is 94M, frozen)")

LoadHead(
  (head): Sequential(
    (0): Dropout(p=0.3, inplace=False)
    (1): Linear(in_features=17, out_features=32, bias=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)
load: 609 trainable parameters (WavLM alone is 94M, frozen)


## 6 — One fold

Early stopping on validation AUC, `pos_weight` for the class prior, threshold chosen on `val`. In the
official protocol there is no validation set, so the run uses a fixed number of epochs and a 0.5
threshold — the paper's operating point.

In [23]:
import copy
import time

import torch
from lib.metrics import best_threshold, evaluate


def train_one_fold(fold, seed=SEED, verbose=True):
    train_ids, val_ids, hold_ids = fold_parts(fold)
    dl_train = loader_for(train_ids, True, seed=seed)
    dl_val = loader_for(val_ids, False) if val_ids is not None else None
    dl_hold = loader_for(hold_ids, False)

    torch.manual_seed(seed)
    net = LoadHead(len(FEATURE_NAMES)).to(DEVICE)
    train_features = torch.stack([dl_train.dataset[i][0] for i in range(len(dl_train.dataset))])
    net.fit_scaler(train_features)          # training fold only: the holdout never reaches it
    optimiser = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=clips.class_weights(train_ids)).to(DEVICE)

    best_score, best_state, best_thr, waited = -1.0, None, 0.5, 0
    started = time.perf_counter()
    epoch = -1
    for epoch in range(EPOCHS):
        net.train()
        for x, y in dl_train:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimiser.zero_grad()
            loss = loss_fn(net(x), y.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimiser.step()

        if dl_val is None:
            continue
        net.eval()
        probs, targets = [], []
        with torch.no_grad():
            for x, y in dl_val:
                probs += torch.sigmoid(net(x.to(DEVICE))).tolist()
                targets += y.tolist()
        score = evaluate(targets, probs)["auc"]
        score = score if score == score else 0.0

        if score > best_score:
            best_score, best_state, waited = score, copy.deepcopy(net.state_dict()), 0
            best_thr = best_threshold(targets, probs)
        else:
            waited += 1
            if waited >= PATIENCE:
                break
    seconds = time.perf_counter() - started

    if best_state is not None:
        net.load_state_dict(best_state)
    net.eval()
    probs, targets = [], []
    with torch.no_grad():
        for x, y in dl_hold:
            probs += torch.sigmoid(net(x.to(DEVICE))).tolist()
            targets += y.tolist()

    ids = dl_hold.dataset.clip_ids
    assert len(ids) == len(probs) == len(targets), (len(ids), len(probs), len(targets))

    metrics = evaluate(targets, probs, threshold=best_thr)
    metrics["seconds"] = seconds
    metrics["epochs"] = epoch + 1
    if verbose:
        print(f"fold {fold} seed {seed}: ACC {metrics['acc']*100:5.1f}  F1 {metrics['f1']*100:5.1f}  "
              f"AUC {metrics['auc']*100:5.1f}  thr {best_thr:.2f}  {seconds:4.0f}s")
    return metrics, ids, targets, probs

## 7 — All folds

The number that goes on the board is the mean over the folds; the spread tells you how much of it is
noise. Three seeds, because a single seed says nothing about seed variance.

In [24]:
import numpy as np
import pandas as pd

print(f"load on {SPLIT}: {len(SEEDS)} seed(s) x {N_FOLDS} folds")

results, pooled = [], []
for seed in SEEDS:
    for fold in range(1, N_FOLDS + 1):
        metrics, ids, targets, probs = train_one_fold(fold, seed=seed)
        results.append({"seed": seed, "fold": fold, **metrics})
        pooled.append(pd.DataFrame({"clip_id": ids, "y_true": targets,
                                    "y_prob": probs, "seed": seed, "fold": fold}))

summary = pd.DataFrame(results)
print(summary[["seed", "fold", "acc", "f1", "auc", "balanced_acc", "seconds"]].round(3)
      .to_string(index=False))
print(f"\nCV  ACC {summary.acc.mean()*100:.1f} ± {summary.acc.std()*100:.1f}  "
      f"F1 {summary.f1.mean()*100:.1f}  AUC {summary.auc.mean()*100:.1f}  "
      f"balanced {summary.balanced_acc.mean()*100:.1f}")

load on official: 3 seed(s) x 3 folds
fold 1 seed 0: ACC  53.0  F1  56.2  AUC  54.8  thr 0.50     3s
fold 2 seed 0: ACC  57.0  F1  62.6  AUC  60.8  thr 0.50     3s
fold 3 seed 0: ACC  56.6  F1  58.1  AUC  60.0  thr 0.50     4s
fold 1 seed 1: ACC  51.7  F1  53.1  AUC  55.5  thr 0.50     3s
fold 2 seed 1: ACC  57.6  F1  62.2  AUC  61.9  thr 0.50     3s
fold 3 seed 1: ACC  55.6  F1  56.8  AUC  60.0  thr 0.50     3s
fold 1 seed 2: ACC  52.2  F1  55.0  AUC  54.7  thr 0.50     3s
fold 2 seed 2: ACC  56.2  F1  61.1  AUC  61.2  thr 0.50     3s
fold 3 seed 2: ACC  55.8  F1  56.8  AUC  60.0  thr 0.50     3s
 seed  fold   acc    f1   auc  balanced_acc  seconds
    0     1 0.530 0.562 0.548         0.528    3.250
    0     2 0.570 0.626 0.608         0.567    3.015
    0     3 0.566 0.581 0.600         0.568    3.619
    1     1 0.517 0.531 0.555         0.518    3.069
    1     2 0.576 0.622 0.619         0.573    3.169
    1     3 0.556 0.568 0.600         0.558    3.330
    2     1 0.522 0.550 

## 8 — Pooled predictions and per-gender breakdown

In [25]:
from lib.metrics import evaluate, majority_baseline

preds = pd.concat(pooled, ignore_index=True)
per_clip = (preds.groupby("clip_id", as_index=False)
                 .agg(y_true=("y_true", "first"), y_prob=("y_prob", "mean")))

overall = evaluate(per_clip.y_true, per_clip.y_prob)
base = majority_baseline(per_clip.y_true)
print(f"pooled ACC {overall['acc']*100:.1f}  balanced {overall['balanced_acc']*100:.1f}  "
      f"AUC {overall['auc']*100:.1f}  | majority {base*100:.1f}")

out = Path("preds"); out.mkdir(exist_ok=True)
target = out / f"cv_{DIRECTION}_{SPLIT}.csv"
per_clip.to_csv(target, index=False)
print("saved", target)

pooled ACC 55.2  balanced 55.0  AUC 58.2  | majority 53.2
saved preds/cv_load_official.csv


In [26]:
meta = pd.read_csv(DATA / "dataset.csv", dtype=str).set_index("clip_id")
per_clip["gender"] = per_clip.clip_id.map(meta.gender)
for gender, group in per_clip.groupby("gender"):
    m = evaluate(group.y_true, group.y_prob)
    print(f"{gender:7} n={m['n']:4d}  ACC {m['acc']*100:5.1f}  balanced {m['balanced_acc']*100:5.1f}")

female  n= 218  ACC  50.9  balanced  49.4
male    n= 914  ACC  56.2  balanced  55.3


## 9 — Row for the results board

Paste it into `## Results board` of `lie-detector/README.md` — that file is shared, so it goes in a
pull request or after telling the group.

In [27]:
from lib.metrics import format_row

board = {"acc": summary.acc.mean(), "acc_std": summary.acc.std(),
         "f1": summary.f1.mean(), "auc": summary.auc.mean(),
         "balanced_acc": summary.balanced_acc.mean()}
row = format_row(DIRECTION, board, owner="<your name>", method="acoustic load",
                 params=params, majority=base, runtime=f"{summary.seconds.mean():.0f} s/fold ({DEVICE})")

if SPLIT == "official":
    print("official 3-fold: clip-level, not comparable to the board. Report it separately.")
else:
    print(row)

official 3-fold: clip-level, not comparable to the board. Report it separately.


## 10 — What this can and cannot show

Read [`docs/theory.md`](docs/theory.md) before interpreting a number.

- A result **above** the `tia` audio branch (58.9 subject AUC, 107k parameters on top of a 94M frozen
  encoder) would say the foundation model is not what carries the signal.
- A result **at chance** says the behavioural cue literature does not transfer to this dataset — which
  would be consistent with the annotation oracle sitting at 51.1 ACC.
- Either way, the direction is **passive**: it measures cues, it does not elicit them. Vrij's own
  evidence is stronger for the eliciting version, which this notebook cannot do.